In [1]:
import os
import time
import pickle
from pathlib import Path

import h5py
import numpy as np
from google.colab import drive


# ============================================================
# Mount Drive
# ============================================================
drive.mount("/content/drive", force_remount=False)


# ============================================================
# Paths / settings
# ============================================================
ROOT = Path(
    "/content/drive/MyDrive/V2_drift/texturemodel/model_fits"
)

SAVE_DIR = Path(
    "/content/drive/MyDrive/V2_drift/texturemodel/drift_results"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = list(range(1, 9))
ROIS = ["V1", "V2", "V3", "hV4"]

NSESSIONS = 30
NPERMS = 1000

# Same seed as SP permutation code
SEED = 1

# Same setting used in SP
FIXED_FIRST = False

# Median across voxels, same as SP
USE_MEDIAN = True

# ============================================================
# MODEL CONFIGURATION
# Change ONLY these values between runs
# ============================================================

PCA_TYPE = "pcaHL"

# 95 = PCA retaining 95% variance
# 0  = no PCA version
PCA_PCT_VAR = 0

# "raw", "zeroMean", or "equalStd"
NORM = "raw"


# Validate settings
if PCA_PCT_VAR not in [0, 95]:
    raise ValueError(
        f"PCA_PCT_VAR must be 0 or 95, got {PCA_PCT_VAR}"
    )

if NORM not in ["raw", "zeroMean", "equalStd"]:
    raise ValueError(
        f"Unknown NORM: {NORM}"
    )


# Filename suffix used by the fitting outputs
if NORM == "raw":
    NORM_SUFFIX = ""
elif NORM == "zeroMean":
    NORM_SUFFIX = "_zeroMean"
elif NORM == "equalStd":
    NORM_SUFFIX = "_equalStd"


print("Configuration:")
print("  PCA_TYPE    =", PCA_TYPE)
print("  PCA_PCT_VAR =", PCA_PCT_VAR)
print("  NORM        =", NORM)

# ============================================================
# Helper: exact TS filename
# ============================================================
def get_ts_file(subject, roi):

    subject_name = f"S{subject:02d}"

    model_name = (
        f"{subject_name}_{roi}_"
        f"texture_pyramid_OLS_4ori_4sf_"
        f"{PCA_TYPE}_{roi}_pca{PCA_PCT_VAR}_"
        f"use_nsd_grid5_pRFs"
    )

    filename = (
        f"{model_name}"
        f"{NORM_SUFFIX}_"
        f"30x30_cross_session.h5py"
    )

    return (
        ROOT
        / subject_name
        / "FAST_CROSS_SESSION"
        / filename
    )


# ============================================================
# PRE-FLIGHT CHECK
# Make sure all 32 files for this configuration exist
# before starting permutations.
# ============================================================

print("\nChecking input files...")

missing_files = []

for subject in SUBJECTS:
    for roi in ROIS:

        fn = get_ts_file(subject, roi)

        if not fn.exists():
            missing_files.append(fn)

if missing_files:

    print("\nMissing files:")
    for fn in missing_files:
        print("  ", fn)

    raise FileNotFoundError(
        f"{len(missing_files)} input files are missing "
        f"for configuration "
        f"{PCA_TYPE}{PCA_PCT_VAR} / {NORM}"
    )

print(
    f"✓ All {len(SUBJECTS) * len(ROIS)} files found "
    f"for {PCA_TYPE}{PCA_PCT_VAR} / {NORM}"
)

# ============================================================
# Session-distance matrices
# Same definitions as SP code
# ============================================================
idx = np.arange(NSESSIONS)

distMatrix = np.abs(
    idx[:, None] - idx[None, :]
)

lagMatrix = (
    idx[None, :] - idx[:, None]
)


# ============================================================
# Generate permutation orders
#
# EXACTLY matches the legacy RNG procedure used in the SP code:
# np.random.seed(1) + np.random.permutation
#
# The same 1000 permutation orders are then used for
# every subject and every ROI.
# ============================================================

np.random.seed(SEED)

perm_orders = np.zeros(
    (NPERMS, NSESSIONS),
    dtype=np.int32
)

for iperm in range(NPERMS):

    if FIXED_FIRST:

        perm_orders[iperm, 0] = 0

        perm_orders[iperm, 1:] = (
            np.random.permutation(
                NSESSIONS - 1
            ) + 1
        )

    else:

        perm_orders[iperm] = (
            np.random.permutation(
                NSESSIONS
            )
        )


print("Permutation orders:", perm_orders.shape)
print("First permutation:")
print(perm_orders[0])


print("Permutation orders:", perm_orders.shape)
print("First permutation:")
print(perm_orders[0])


# ============================================================
# Allocate outputs
#
# Layout parallels the useful parts of save_perms_expand
# ============================================================

nsubjects = len(SUBJECTS)
nrois = len(ROIS)


# Observed ROI-level 30x30 matrices
r2split = np.full(
    (nsubjects, nrois, NSESSIONS, NSESSIONS),
    np.nan,
    dtype=np.float32
)

pearsonR = np.full_like(
    r2split,
    np.nan
)


# Observed lag means: lag 1..29
r2Dist = np.full(
    (nsubjects, nrois, NSESSIONS - 1),
    np.nan,
    dtype=np.float32
)

pearsonDist = np.full_like(
    r2Dist,
    np.nan
)


# Symmetrized individual session pairs for each lag
#
# Stored as nested:
# [subject][roi][lag]
#
# Each vector has length 30-lag.
r2DistSess = [
    [
        [
            np.full(
                NSESSIONS - lag,
                np.nan,
                dtype=np.float32
            )
            for lag in range(1, NSESSIONS)
        ]
        for roi in ROIS
    ]
    for subject in SUBJECTS
]

pearsonDistSess = [
    [
        [
            np.full(
                NSESSIONS - lag,
                np.nan,
                dtype=np.float32
            )
            for lag in range(1, NSESSIONS)
        ]
        for roi in ROIS
    ]
    for subject in SUBJECTS
]


# Permuted lag means
r2DistPerm = np.full(
    (
        nsubjects,
        nrois,
        NPERMS,
        NSESSIONS - 1
    ),
    np.nan,
    dtype=np.float32
)

pearsonDistPerm = np.full_like(
    r2DistPerm,
    np.nan
)


# Full permuted matrices.
#
# The SP code stored these as well, so we retain them.
r2perm = np.full(
    (
        nsubjects,
        nrois,
        NPERMS,
        NSESSIONS,
        NSESSIONS
    ),
    np.nan,
    dtype=np.float32
)

pearsonPerm = np.full_like(
    r2perm,
    np.nan
)


# QC / metadata
totalVox = np.zeros(
    (nsubjects, nrois),
    dtype=np.int32
)

numValidVox = np.zeros_like(
    totalVox
)

excludedNonfiniteVox = np.zeros_like(
    totalVox
)


# ============================================================
# MAIN LOOP
# ============================================================
t0 = time.time()

for isub_idx, subject in enumerate(SUBJECTS):

    print("\n" + "=" * 80)
    print(f"SUBJECT {subject}")
    print("=" * 80)

    for iroi, roi in enumerate(ROIS):

        fn = get_ts_file(subject, roi)

        print(f"\n{roi}")
        print(fn)

        if not fn.exists():
            raise FileNotFoundError(fn)

        # ----------------------------------------------------
        # Load cross-session values
        # ----------------------------------------------------
        with h5py.File(fn, "r") as f:

            r2 = np.asarray(
                f["val_r2"],
                dtype=np.float32
            )[..., 0]

            cc = np.asarray(
                f["val_cc"],
                dtype=np.float32
            )[..., 0]

            completed = np.asarray(
                f["completed_train_sessions"],
                dtype=bool
            )

        if r2.shape[:2] != (30, 30):
            raise RuntimeError(
                f"{subject} {roi}: unexpected R2 shape "
                f"{r2.shape}"
            )

        if cc.shape != r2.shape:
            raise RuntimeError(
                f"{subject} {roi}: CC/R2 shape mismatch"
            )

        if completed.sum() != 30:
            raise RuntimeError(
                f"{subject} {roi}: "
                f"only {completed.sum()}/30 train sessions complete"
            )

        nvox = r2.shape[2]

        totalVox[isub_idx, iroi] = nvox


        # ----------------------------------------------------
        # FIXED voxel population
        #
        # A voxel is retained only if both R2 and Pearson are
        # finite for every train x test session combination.
        #
        # This removes the small S06 session-4 issue without
        # changing voxel composition as a function of lag.
        # ----------------------------------------------------
        valid_vox = (
            np.all(np.isfinite(r2), axis=(0, 1))
            &
            np.all(np.isfinite(cc), axis=(0, 1))
        )

        nvalid = int(valid_vox.sum())

        numValidVox[isub_idx, iroi] = nvalid
        excludedNonfiniteVox[isub_idx, iroi] = (
            nvox - nvalid
        )

        print(
            f"voxels: {nvox} total, "
            f"{nvalid} retained, "
            f"{nvox - nvalid} excluded"
        )

        if nvalid == 0:
            raise RuntimeError(
                f"{subject} {roi}: no valid voxels"
            )

        r2 = r2[:, :, valid_vox]
        cc = cc[:, :, valid_vox]


        # ----------------------------------------------------
        # Median across voxels
        #
        # Exact analogue of SP:
        #
        # np.nanmedian(
        #     roiNsdR2[:use, goodVox_mask, :use],
        #     axis=1
        # )
        #
        # TS dimensions are train x test x voxel,
        # therefore voxel axis = 2.
        # ----------------------------------------------------
        if USE_MEDIAN:

            r2_mat = np.nanmedian(
                r2,
                axis=2
            )

            cc_mat = np.nanmedian(
                cc,
                axis=2
            )

        else:

            r2_mat = np.nanmean(
                r2,
                axis=2
            )

            cc_mat = np.nanmean(
                cc,
                axis=2
            )


        r2split[
            isub_idx,
            iroi
        ] = r2_mat

        pearsonR[
            isub_idx,
            iroi
        ] = cc_mat


        # ====================================================
        # OBSERVED DISTANCE METRICS
        #
        # Same calculations as SP code
        # ====================================================
        for lag in range(1, NSESSIONS):

            mask_pos = (
                lagMatrix == lag
            )

            mask_neg = (
                lagMatrix == -lag
            )

            # ---------------------------------------------
            # R2
            # ---------------------------------------------
            pair_values_r2 = 0.5 * (
                r2_mat[mask_pos]
                +
                r2_mat[mask_neg]
            )

            r2DistSess[
                isub_idx
            ][
                iroi
            ][
                lag - 1
            ][:] = pair_values_r2

            r2Dist[
                isub_idx,
                iroi,
                lag - 1
            ] = np.nanmean(
                r2_mat[
                    distMatrix == lag
                ]
            )


            # ---------------------------------------------
            # Pearson
            # ---------------------------------------------
            pair_values_cc = 0.5 * (
                cc_mat[mask_pos]
                +
                cc_mat[mask_neg]
            )

            pearsonDistSess[
                isub_idx
            ][
                iroi
            ][
                lag - 1
            ][:] = pair_values_cc

            pearsonDist[
                isub_idx,
                iroi,
                lag - 1
            ] = np.nanmean(
                cc_mat[
                    distMatrix == lag
                ]
            )


        # ====================================================
        # PERMUTATIONS
        # ====================================================
        for iperm in range(NPERMS):

            permOrder = perm_orders[iperm]

            # Same permutation on BOTH axes.
            temp_r2 = r2_mat[
                permOrder[:, None],
                permOrder[None, :]
            ]

            temp_cc = cc_mat[
                permOrder[:, None],
                permOrder[None, :]
            ]

            r2perm[
                isub_idx,
                iroi,
                iperm
            ] = temp_r2

            pearsonPerm[
                isub_idx,
                iroi,
                iperm
            ] = temp_cc


            # -----------------------------------------------
            # Lag summaries of this permutation
            # -----------------------------------------------
            for lag in range(1, NSESSIONS):

                r2DistPerm[
                    isub_idx,
                    iroi,
                    iperm,
                    lag - 1
                ] = np.nanmean(
                    temp_r2[
                        distMatrix == lag
                    ]
                )

                pearsonDistPerm[
                    isub_idx,
                    iroi,
                    iperm,
                    lag - 1
                ] = np.nanmean(
                    temp_cc[
                        distMatrix == lag
                    ]
                )


        print("  done")


# ============================================================
# Save
# ============================================================

out_dict = {

    # configuration
    "subjects": SUBJECTS,
    "rois": ROIS,
    "nsessions": NSESSIONS,
    "nperms": NPERMS,
    "seed": SEED,
    "fixedFirst": FIXED_FIRST,
    "useMedian": USE_MEDIAN,

    "model": "texture_pyramid",
    "analysis_level": "single_voxel",
    "pca_type": PCA_TYPE,
    "pca_pct_var": PCA_PCT_VAR,
    "normalization": NORM,

    "pca_applied": PCA_PCT_VAR != 0,

    "n_ori": 4,
    "n_sf": 4,
    "ridge": 0,
    "which_prf_grid": 5,

    # permutations
    "permOrders": perm_orders,

    # voxel QC
    "totalVox": totalVox,
    "numValidVox": numValidVox,
    "excludedNonfiniteVox": excludedNonfiniteVox,

    # observed matrices
    "r2split": r2split,
    "pearsonR": pearsonR,

    # observed lag summaries
    "r2Dist": r2Dist,
    "pearsonDist": pearsonDist,

    "r2DistSess": r2DistSess,
    "pearsonDistSess": pearsonDistSess,

    # permutations
    "r2perm": r2perm,
    "pearsonPerm": pearsonPerm,

    "r2DistPerm": r2DistPerm,
    "pearsonDistPerm": pearsonDistPerm,
}


config_tag = (
    f"{PCA_TYPE}{PCA_PCT_VAR}_{NORM}"
)

out_path = (
    SAVE_DIR
    / (
        f"perm_single_voxel_TS_"
        f"{config_tag}_"
        f"4ori_4sf_grid5_OLS_"
        f"N{NPERMS}.pkl"
    )
)

with open(out_path, "wb") as f:
    pickle.dump(
        out_dict,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


elapsed = time.time() - t0

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("Saved:")
print(out_path)

print(
    f"Runtime: {elapsed / 60:.1f} minutes"
)

print("\nValid voxel counts:")
for isub_idx, subject in enumerate(SUBJECTS):
    print(
        f"S{subject:02d}:",
        {
            roi: int(
                numValidVox[
                    isub_idx,
                    iroi
                ]
            )
            for iroi, roi in enumerate(ROIS)
        }
    )

Mounted at /content/drive
Configuration:
  PCA_TYPE    = pcaHL
  PCA_PCT_VAR = 0
  NORM        = raw

Checking input files...
✓ All 32 files found for pcaHL0 / raw
Permutation orders: (1000, 30)
First permutation:
[17 21 10 19 14 20 26  3 24 22 23  4  2 25  6 18 13  7 27  1 16  0 15 29
 28  9  8 12 11  5]
Permutation orders: (1000, 30)
First permutation:
[17 21 10 19 14 20 26  3 24 22 23  4  2 25  6 18 13  7 27  1 16  0 15 29
 28  9  8 12 11  5]

SUBJECT 1

V1
/content/drive/MyDrive/V2_drift/texturemodel/model_fits/S01/FAST_CROSS_SESSION/S01_V1_texture_pyramid_OLS_4ori_4sf_pcaHL_V1_pca0_use_nsd_grid5_pRFs_30x30_cross_session.h5py
voxels: 1350 total, 1350 retained, 0 excluded
  done

V2
/content/drive/MyDrive/V2_drift/texturemodel/model_fits/S01/FAST_CROSS_SESSION/S01_V2_texture_pyramid_OLS_4ori_4sf_pcaHL_V2_pca0_use_nsd_grid5_pRFs_30x30_cross_session.h5py
voxels: 1433 total, 1433 retained, 0 excluded
  done

V3
/content/drive/MyDrive/V2_drift/texturemodel/model_fits/S01/FAST_CROSS_SESS

In [ ]:
import pickle
import numpy as np
from pathlib import Path


# ============================================================
# File to check
# ============================================================
FN = Path(
    "/content/drive/MyDrive/V2_drift/texturemodel/drift_results/"
    "perm_single_voxel_TS_pcaHL95_raw_4ori_4sf_grid5_OLS_N1000.pkl"
)

assert FN.exists(), f"File not found:\n{FN}"

print("Loading:")
print(FN)


# ============================================================
# Load
# ============================================================
with open(FN, "rb") as f:
    d = pickle.load(f)


print("\n" + "=" * 80)
print("METADATA")
print("=" * 80)

for key in [
    "analysis_level",
    "model",
    "pca_type",
    "pca_pct_var",
    "normalization",
    "n_ori",
    "n_sf",
    "ridge",
    "which_prf_grid",
    "nsessions",
    "nperms",
    "seed",
    "fixedFirst",
    "useMedian",
]:
    print(f"{key:20s}: {d.get(key)}")


# ============================================================
# Expected configuration
# ============================================================
assert d["analysis_level"] == "single_voxel"
assert d["model"] == "texture_pyramid"
assert d["pca_type"] == "pcaHL"
assert d["pca_pct_var"] == 95
assert d["normalization"] == "raw"

assert d["nsessions"] == 30
assert d["nperms"] == 1000
assert d["seed"] == 1
assert d["fixedFirst"] is False

print("\n✓ Metadata correct")


# ============================================================
# Check array shapes
# ============================================================
print("\n" + "=" * 80)
print("ARRAY SHAPES")
print("=" * 80)

expected_shapes = {
    "permOrders":       (1000, 30),

    "r2split":          (8, 4, 30, 30),
    "pearsonR":         (8, 4, 30, 30),

    "r2Dist":           (8, 4, 29),
    "pearsonDist":      (8, 4, 29),

    "r2perm":           (8, 4, 1000, 30, 30),
    "pearsonPerm":      (8, 4, 1000, 30, 30),

    "r2DistPerm":       (8, 4, 1000, 29),
    "pearsonDistPerm":  (8, 4, 1000, 29),

    "totalVox":         (8, 4),
    "numValidVox":      (8, 4),
}

for key, expected in expected_shapes.items():

    actual = np.asarray(d[key]).shape

    print(
        f"{key:20s}: "
        f"{str(actual):25s} "
        f"expected {expected}"
    )

    assert actual == expected, (
        f"{key}: expected {expected}, got {actual}"
    )

print("\n✓ All array shapes correct")


# ============================================================
# Verify legacy SP permutation RNG EXACTLY
# ============================================================
print("\n" + "=" * 80)
print("LEGACY SP RNG CHECK")
print("=" * 80)

saved_orders = np.asarray(d["permOrders"])

# Reproduce exactly:
# np.random.seed(1)
# np.random.permutation(30)
np.random.seed(1)

expected_orders = np.zeros(
    (1000, 30),
    dtype=np.int32
)

for iperm in range(1000):
    expected_orders[iperm] = np.random.permutation(30)


print("Saved first permutation:")
print(saved_orders[0])

print("\nExpected SP first permutation:")
print(expected_orders[0])


assert np.array_equal(
    saved_orders,
    expected_orders
), (
    "Permutation orders DO NOT match "
    "np.random.seed(1) + np.random.permutation!"
)

print(
    "\n✓ All 1000 permutation orders exactly match "
    "the legacy SP RNG procedure"
)


# ============================================================
# Verify every row is actually a permutation of 0..29
# ============================================================
target = np.arange(30)

for iperm in range(1000):

    assert np.array_equal(
        np.sort(saved_orders[iperm]),
        target
    ), f"Invalid permutation at index {iperm}"

print("✓ All 1000 rows are valid permutations of sessions 0–29")


# ============================================================
# Check voxel counts
# ============================================================
print("\n" + "=" * 80)
print("VOXEL QC")
print("=" * 80)

ROIS = d["rois"]

for s in range(8):

    print(f"\nS{s+1:02d}")

    for r, roi in enumerate(ROIS):

        total = int(d["totalVox"][s, r])
        valid = int(d["numValidVox"][s, r])
        excluded = int(
            d["excludedNonfiniteVox"][s, r]
        )

        print(
            f"  {roi:3s}: "
            f"{valid}/{total} retained "
            f"({excluded} excluded)"
        )

        assert total == valid + excluded

print("\n✓ Voxel counts internally consistent")


# ============================================================
# Check for unexpected NaN / Inf
# ============================================================
print("\n" + "=" * 80)
print("FINITE VALUE CHECK")
print("=" * 80)

for key in [
    "r2split",
    "pearsonR",
    "r2Dist",
    "pearsonDist",
    "r2perm",
    "pearsonPerm",
    "r2DistPerm",
    "pearsonDistPerm",
]:

    arr = np.asarray(d[key])

    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()

    print(
        f"{key:20s}: "
        f"NaN={n_nan:,}   "
        f"Inf={n_inf:,}"
    )

    assert n_inf == 0
    assert n_nan == 0

print("\n✓ No NaN or Inf values in analysis outputs")


# ============================================================
# Strong check:
# verify stored permutation #0 is really the observed
# matrix reordered using permOrders[0]
# ============================================================
print("\n" + "=" * 80)
print("PERMUTED MATRIX CHECK")
print("=" * 80)

p = saved_orders[0]

for s in range(8):

    for r in range(4):

        expected_r2 = d["r2split"][s, r][
            p[:, None],
            p[None, :]
        ]

        expected_cc = d["pearsonR"][s, r][
            p[:, None],
            p[None, :]
        ]

        assert np.allclose(
            d["r2perm"][s, r, 0],
            expected_r2,
            equal_nan=True
        )

        assert np.allclose(
            d["pearsonPerm"][s, r, 0],
            expected_cc,
            equal_nan=True
        )

print(
    "✓ Stored permutation matrices correctly use "
    "the same session permutation on both axes"
)


# ============================================================
# Final result
# ============================================================
print("\n" + "=" * 80)
print("ALL CHECKS PASSED ✓")
print("=" * 80)

print(
    "The pcaHL95/raw single-voxel permutation file is "
    "internally consistent and uses the legacy SP "
    "np.random.seed(1) + np.random.permutation procedure."
)

Loading:
/content/drive/MyDrive/V2_drift/texturemodel/drift_results/perm_single_voxel_TS_pcaHL95_raw_4ori_4sf_grid5_OLS_N1000.pkl

METADATA
analysis_level      : single_voxel
model               : texture_pyramid
pca_type            : pcaHL
pca_pct_var         : 95
normalization       : raw
n_ori               : 4
n_sf                : 4
ridge               : 0
which_prf_grid      : 5
nsessions           : 30
nperms              : 1000
seed                : 1
fixedFirst          : False
useMedian           : True

✓ Metadata correct

ARRAY SHAPES
permOrders          : (1000, 30)                expected (1000, 30)
r2split             : (8, 4, 30, 30)            expected (8, 4, 30, 30)
pearsonR            : (8, 4, 30, 30)            expected (8, 4, 30, 30)
r2Dist              : (8, 4, 29)                expected (8, 4, 29)
pearsonDist         : (8, 4, 29)                expected (8, 4, 29)
r2perm              : (8, 4, 1000, 30, 30)      expected (8, 4, 1000, 30, 30)
pearsonPerm         :